# Equal RRF Weights + Gemini Agents (No LLM Judge) — UC1, UC2, UC3

Runs the same retrieval + evaluation flow as `3_evaluation_query_extension_uc1.ipynb`, but:

- **Gemini** generates the four query variants once into `output_with_agents_<uc>.csv`
- **No LLM judge** (`enable_llm_judge=False`)
- **Equal weighted RRF**: `[0.2, 0.2, 0.2, 0.2, 0.2]` for baseline + four agent variants
- Applies to **uc1**, **uc2**, and **uc3**

**Resume behavior:** Step 1 only calls Gemini for rows/columns that are still missing (empty or still equal to the original query). Step 2 reuses those cached variants and only reruns BM25 + RRF + cross-encoder (no LLM calls).

The custom-weights notebook reuses the same `output_with_agents_*.csv` files; run Step 1 in either notebook only until all variants are complete.

In [2]:
from pathlib import Path
import importlib
import os

import numpy as np
import pandas as pd

project_root = Path.cwd()
while project_root.name != "Legal-Query-Synthesis-from-Business-Processes-for-Agentic-Retrieval" and project_root.parent != project_root:
    project_root = project_root.parent

os.chdir(project_root)
print(f"Working directory: {Path.cwd()}")

Working directory: /Users/mareklorenz/Development/Legal-Query-Synthesis-from-Business-Processes-for-Agentic-Retrieval


In [3]:
import main

USE_CASES = ["uc1", "uc2", "uc3"]
LLM_PROVIDER = "gemini"
ENABLE_LLM_JUDGE = False
WEIGHTED_RRF_WEIGHTS = [0.2, 0.2, 0.2, 0.2, 0.2]
EXPERIMENT_LABEL = "equal_rrf_0.2_gemini_no_judge"

main.WEIGHTED_RRF_WEIGHTS = WEIGHTED_RRF_WEIGHTS
importlib.reload(main)
main.WEIGHTED_RRF_WEIGHTS = WEIGHTED_RRF_WEIGHTS

print(f"Provider: {LLM_PROVIDER}")
print(f"LLM judge: {ENABLE_LLM_JUDGE}")
print(f"Weighted RRF weights: {main.WEIGHTED_RRF_WEIGHTS}")

Provider: gemini
LLM judge: False
Weighted RRF weights: [0.2, 0.2, 0.2, 0.2, 0.2]


In [4]:
# Step 1: Gemini only for missing query variants (safe to rerun after quota errors)
for use_case in USE_CASES:
    print(f"\n=== Ensuring Gemini query variants for {use_case.upper()} ===")
    main.ensure_query_variants(
        use_case=use_case,
        provider=LLM_PROVIDER,
        fill_missing_only=True,
    )

# Step 2: Retrieval uses cached variants — no Gemini calls (only RRF weight changes between notebooks)
for use_case in USE_CASES:
    print(f"\n=== Running retrieval pipeline for {use_case.upper()} (records only) ===")
    main.retrieval_pipeline(
        provider="records",
        use_case=use_case,
        enable_llm_judge=ENABLE_LLM_JUDGE,
        skip_variant_generation=True,
    )


=== Ensuring Gemini query variants for UC1 ===
Loaded 1 queries for level 'process'
Loaded 7 queries for level 'subprocess'
Loaded 31 queries for level 'task'
[uc1 / task] Generating variants for: After the customer filled out the claim form, they need to sent it to the insurance compan...
  Missing columns: legal_terminology_rewrite
  Generating missing variant: legal_terminology_rewrite
Saved query variants to output_with_agents_uc1.csv

=== Ensuring Gemini query variants for UC2 ===
Loaded 1 queries for level 'process'
Loaded 7 queries for level 'subprocess'
Loaded 19 queries for level 'task'
[uc2 / task] Generating variants for: Set up the customer's account in the bank's system, including any necessary product or ser...
  Missing columns: legal_terminology_rewrite, contract_clause_query, risk_scenario_query
  Generating missing variant: legal_terminology_rewrite
  Generating missing variant: contract_clause_query
  Generating missing variant: risk_scenario_query
[ERROR] Gemini ge

In [5]:
def clean_text(text):
    cleaned_text = str(text).replace("or\n\n\n", " ")
    cleaned_text = cleaned_text.replace("or\n\n", " ")
    cleaned_text = cleaned_text.replace("and\n\n\n", " ")
    cleaned_text = cleaned_text.replace("and\n\n", " ")
    cleaned_text = cleaned_text.replace("\n\n\n", " ")
    cleaned_text = cleaned_text.replace("\n\n", " ")
    cleaned_text = cleaned_text.replace("\n \n", " ")
    cleaned_text = cleaned_text.replace("\n", " ")
    return cleaned_text


def evaluate_with_original_logic(df_gs, df_alg):
    df_gs = df_gs.copy()
    df_alg = df_alg.copy()

    df_gs["query_cleaned"] = df_gs.apply(lambda row: clean_text(row["query"]), axis=1)
    df_gs["rel_text_cleaned"] = df_gs.apply(lambda row: clean_text(row["rel_text"]), axis=1)
    df_gs = df_gs.drop(["query", "rel_text"], axis=1)
    df_gs = df_gs.rename(columns={"query_cleaned": "query", "rel_text_cleaned": "rel_text"})

    df_alg["query"] = df_alg["query"].apply(clean_text)
    df_alg["rel_text"] = df_alg["rel_text"].apply(clean_text)
    df_alg["rank"] = df_alg.groupby("query")["score"].rank(ascending=False)

    df_gs_enhanced = pd.merge(
        df_gs,
        df_alg,
        how="left",
        left_on=["query", "rel_text"],
        right_on=["query", "rel_text"],
    )

    df_gs_enhanced["AP"] = 1 / df_gs_enhanced["rank"]
    df_gs_enhanced = df_gs_enhanced.fillna(0)

    map_value = df_gs_enhanced["AP"].mean()
    tp_per_query = df_gs_enhanced.groupby("query")["rank"].apply(lambda x: (x > 0).sum()).reset_index(name="count")
    fn_per_query = df_gs_enhanced.groupby("query")["rank"].apply(lambda x: (x == 0).sum()).reset_index(name="count")

    return {
        "map": map_value,
        "avg_tp": tp_per_query["count"].mean(),
        "avg_fn": fn_per_query["count"].mean(),
        "details": df_gs_enhanced,
    }

In [6]:
RUN_SHEETS = {
    "bm25_baseline": ("bm25", "BM25_baseline"),
    "bm25_rrf_query_merge": ("bm25", "BM25_RRF_query_merge"),
    "bm25_rrf_weighted": ("bm25", "BM25_RRF_weighted"),
    "bm25_ce": ("finished_pipeline", "BM25_CE"),
    "bm25_rrf_ce": ("finished_pipeline", "BM25_RRF_CE"),
    "bm25_rrf_weighted_ce": ("finished_pipeline", "BM25_RRF_weighted_CE"),
}

LEVEL_GS_FILES = {
    "process": "process_level",
    "subprocess": "subprocess_level",
    "task": "event_level",
}


def gs_path_for(use_case, level):
    suffix = LEVEL_GS_FILES[level]
    return (
        project_root
        / f"regulatory_relevance4process/SOTA_NLP_LIR/output_ranking_input_eval/{use_case}/gold_standard/gs_{use_case}_{suffix}.xlsx"
    )


def read_output_sheet(use_case, sheet_name):
    ranking_path = project_root / f"output_ranking_{use_case}.xlsx"
    try:
        return pd.read_excel(ranking_path, sheet_name=sheet_name)
    except ValueError:
        return pd.DataFrame(columns=["level", "query", "rel_text", "score"])


def add_level_if_missing(use_case, df):
    if "level" in df.columns:
        return df

    df = df.copy()
    process_queries = set(pd.read_excel(gs_path_for(use_case, "process"))["query"].astype(str).apply(clean_text).tolist())
    subprocess_queries = set(
        pd.read_excel(
            project_root
            / f"regulatory_relevance4process/SOTA_NLP_LIR/input_ranking/{use_case}/Input_queries_medium_{use_case}.xlsx"
        )["process_text"]
        .astype(str)
        .apply(clean_text)
        .tolist()
    )
    task_queries = set(
        pd.read_excel(
            project_root
            / f"regulatory_relevance4process/SOTA_NLP_LIR/input_ranking/{use_case}/Input_queries_low_{use_case}.xlsx"
        )["process_text"]
        .astype(str)
        .apply(clean_text)
        .tolist()
    )

    def infer_level(query):
        q = clean_text(query)
        if q in task_queries:
            return "task"
        if q in subprocess_queries:
            return "subprocess"
        if q in process_queries:
            return "process"
        return "unknown"

    df["level"] = df["query"].apply(infer_level)
    return df[df["level"] != "unknown"].copy()


def evaluate_use_case(use_case):
    rows = []
    for level in ["process", "subprocess", "task"]:
        df_gs = pd.read_excel(gs_path_for(use_case, level))
        for run_name, (stage, sheet_name) in RUN_SHEETS.items():
            df_run = add_level_if_missing(use_case, read_output_sheet(use_case, sheet_name))
            if df_run.empty:
                continue
            df_level = df_run[df_run["level"] == level].copy()
            run_eval = evaluate_with_original_logic(df_gs, df_level)
            rows.append(
                {
                    "experiment": EXPERIMENT_LABEL,
                    "use_case": use_case,
                    "stage": stage,
                    "level": level,
                    "run": run_name,
                    "MAP": run_eval["map"],
                    "avg_true_positives": run_eval["avg_tp"],
                    "avg_false_negatives": run_eval["avg_fn"],
                }
            )
    return pd.DataFrame(rows)


all_comparison_frames = [evaluate_use_case(use_case) for use_case in USE_CASES]
comparison = pd.concat(all_comparison_frames, ignore_index=True)
comparison

,experiment,use_case,stage,level,run,MAP,avg_true_positives,avg_false_negatives
0,equal_rrf_0.2_gemini_no_judge,uc1,bm25,process,bm25_baseline,0.046035,14.000000,35.000000
1,equal_rrf_0.2_gemini_no_judge,uc1,bm25,process,bm25_rrf_query_merge,0.029059,15.000000,34.000000
2,equal_rrf_0.2_gemini_no_judge,uc1,bm25,process,bm25_rrf_weighted,0.042120,14.000000,35.000000
3,equal_rrf_0.2_gemini_no_judge,uc1,finished_pipeline,process,bm25_ce,0.022025,27.000000,22.000000
4,equal_rrf_0.2_gemini_no_judge,uc1,finished_pipeline,process,bm25_rrf_ce,0.022657,15.000000,34.000000
5,equal_rrf_0.2_gemini_no_judge,uc1,finished_pipeline,process,bm25_rrf_weighted_ce,0.021072,14.000000,35.000000
6,equal_rrf_0.2_gemini_no_judge,uc1,bm25,subprocess,bm25_baseline,0.036449,1.428571,9.142857
7,equal_rrf_0.2_gemini_no_judge,uc1,bm25,subprocess,bm25_rrf_query_merge,0.008436,0.714286,9.857143
8,equal_rrf_0.2_gemini_no_judge,uc1,bm25,subprocess,bm25_rrf_weighted,0.014362,1.571429,9.000000
9,equal_rrf_0.2_gemini_no_judge,uc1,finished_pipeline,subprocess,bm25_ce,0.057928,3.428571,7.142857


In [7]:
weighted_comparison = comparison[
    comparison["run"].isin(["bm25_rrf_weighted", "bm25_rrf_weighted_ce"])
].copy()
weighted_comparison[["MAP", "avg_true_positives", "avg_false_negatives"]] = weighted_comparison[
    ["MAP", "avg_true_positives", "avg_false_negatives"]
].round(6)
weighted_comparison

,experiment,use_case,stage,level,run,MAP,avg_true_positives,avg_false_negatives
2,equal_rrf_0.2_gemini_no_judge,uc1,bm25,process,bm25_rrf_weighted,0.042120,14.000000,35.000000
5,equal_rrf_0.2_gemini_no_judge,uc1,finished_pipeline,process,bm25_rrf_weighted_ce,0.021072,14.000000,35.000000
8,equal_rrf_0.2_gemini_no_judge,uc1,bm25,subprocess,bm25_rrf_weighted,0.014362,1.571429,9.000000
11,equal_rrf_0.2_gemini_no_judge,uc1,finished_pipeline,subprocess,bm25_rrf_weighted_ce,0.043814,1.571429,9.000000
14,equal_rrf_0.2_gemini_no_judge,uc1,bm25,task,bm25_rrf_weighted,0.065488,0.827586,3.758621
17,equal_rrf_0.2_gemini_no_judge,uc1,finished_pipeline,task,bm25_rrf_weighted_ce,0.107075,0.827586,3.758621
20,equal_rrf_0.2_gemini_no_judge,uc2,bm25,process,bm25_rrf_weighted,0.048694,14.000000,17.000000
23,equal_rrf_0.2_gemini_no_judge,uc2,finished_pipeline,process,bm25_rrf_weighted_ce,0.032871,14.000000,17.000000
26,equal_rrf_0.2_gemini_no_judge,uc2,bm25,subprocess,bm25_rrf_weighted,0.030801,1.571429,7.000000
29,equal_rrf_0.2_gemini_no_judge,uc2,finished_pipeline,subprocess,bm25_rrf_weighted_ce,0.046022,1.571429,7.000000


In [8]:
PAPER_METHODS = {
    "BM25+CE + weighted query diversification": "BM25_RRF_weighted_CE",
}

LEVEL_LABELS = {
    "process": "level 1: process relevance",
    "subprocess": "level 2: sub-process relevance",
    "task": "level 3: task/event relevance",
}

USE_CASE_LABELS = {
    "uc1": "use case 1",
    "uc2": "use case 2",
    "uc3": "use case 3",
}


def load_paper_inputs(use_case, sheet_name):
    ranking_path = project_root / f"output_ranking_{use_case}.xlsx"
    corpus_path = (
        project_root
        / f"regulatory_relevance4process/SOTA_NLP_LIR/input_ranking/{use_case}/Input_corpus_{use_case}.xlsx"
    )
    if not ranking_path.exists():
        return None, None
    try:
        df_predictions = pd.read_excel(ranking_path, sheet_name=sheet_name)
    except ValueError:
        return None, None
    df_corpus = pd.read_excel(corpus_path)
    corpus_texts = set(df_corpus["requirement_text"].astype(str).apply(clean_text).tolist())
    return df_predictions, corpus_texts


def relevance_metrics_for_level(use_case, level, sheet_name):
    df_predictions, corpus_texts = load_paper_inputs(use_case, sheet_name)
    if df_predictions is None:
        return {"Acc.": np.nan, "Prec.": np.nan, "Rec.": np.nan}

    gs_path = gs_path_for(use_case, level)
    df_gold = pd.read_excel(gs_path)
    df_gold = df_gold.copy()
    df_predictions = df_predictions.copy()
    df_gold["query"] = df_gold["query"].apply(clean_text)
    df_gold["rel_text"] = df_gold["rel_text"].apply(clean_text)
    df_predictions["query"] = df_predictions["query"].apply(clean_text)
    df_predictions["rel_text"] = df_predictions["rel_text"].apply(clean_text)
    if "level" in df_predictions.columns:
        df_predictions = df_predictions[df_predictions["level"] == level].copy()

    queries = sorted(set(df_gold["query"].tolist()) | set(df_predictions["query"].tolist()))
    true_positives = false_positives = false_negatives = true_negatives = 0
    for query in queries:
        gold_relevant = set(df_gold[df_gold["query"] == query]["rel_text"].tolist())
        predicted_relevant = set(df_predictions[df_predictions["query"] == query]["rel_text"].tolist())
        true_positives += len(gold_relevant & predicted_relevant)
        false_positives += len(predicted_relevant - gold_relevant)
        false_negatives += len(gold_relevant - predicted_relevant)
        true_negatives += len(corpus_texts - gold_relevant - predicted_relevant)

    accuracy_denominator = true_positives + false_positives + false_negatives + true_negatives
    precision_denominator = true_positives + false_positives
    recall_denominator = true_positives + false_negatives
    accuracy = (true_positives + true_negatives) / accuracy_denominator if accuracy_denominator else np.nan
    precision = true_positives / precision_denominator if precision_denominator else np.nan
    recall = true_positives / recall_denominator if recall_denominator else np.nan
    return {"Acc.": accuracy, "Prec.": precision, "Rec.": recall}


paper_rows = []
for level in ["process", "subprocess", "task"]:
    for method_name, sheet_name in PAPER_METHODS.items():
        row = {
            ("", "process level"): LEVEL_LABELS[level],
            ("", "method"): method_name,
        }
        for use_case, use_case_label in USE_CASE_LABELS.items():
            metrics = relevance_metrics_for_level(use_case, level, sheet_name)
            for metric_name, metric_value in metrics.items():
                row[(use_case_label, metric_name)] = metric_value
        paper_rows.append(row)

paper_table = pd.DataFrame(paper_rows)
paper_table.columns = pd.MultiIndex.from_tuples(paper_table.columns)
metric_columns = [
    (use_case_label, metric_name)
    for use_case_label in USE_CASE_LABELS.values()
    for metric_name in ["Acc.", "Prec.", "Rec."]
]
paper_table[metric_columns] = paper_table[metric_columns].round(2)
paper_table

\
                    process level                                    method   
0      level 1: process relevance  BM25+CE + weighted query diversification   
1  level 2: sub-process relevance  BM25+CE + weighted query diversification   
2   level 3: task/event relevance  BM25+CE + weighted query diversification   

  use case 1             use case 2             use case 3              
        Acc. Prec.  Rec.       Acc. Prec.  Rec.       Acc. Prec.  Rec.  
0       0.77  0.14  0.29       0.72  0.14  0.45       0.48  0.04  0.24  
1       0.92  0.05  0.15       0.89  0.05  0.18       0.79  0.01  0.02  
2       0.96  0.05  0.18       0.94  0.04  0.10       0.89  0.00  0.01

In [9]:
export_path = project_root / f"evaluation_{EXPERIMENT_LABEL}.xlsx"
with pd.ExcelWriter(export_path) as writer:
    comparison.to_excel(writer, sheet_name="map_comparison", index=False)
    weighted_comparison.to_excel(writer, sheet_name="weighted_runs", index=False)
    paper_table.to_excel(writer, sheet_name="paper_metrics")
    pd.DataFrame(
        {
            "experiment": [EXPERIMENT_LABEL],
            "provider": [LLM_PROVIDER],
            "enable_llm_judge": [ENABLE_LLM_JUDGE],
            "weighted_rrf_weights": [str(WEIGHTED_RRF_WEIGHTS)],
            "use_cases": [",".join(USE_CASES)],
        }
    ).to_excel(writer, sheet_name="config", index=False)
print(f"Exported {export_path}")

Exported /Users/mareklorenz/Development/Legal-Query-Synthesis-from-Business-Processes-for-Agentic-Retrieval/evaluation_equal_rrf_0.2_gemini_no_judge.xlsx
